# HU-4 · Análisis exploratorio (EDA) y limpieza de datos
**Proyecto BIODATAPHARMA · Grupo Arthena · Artritis reumatoide**

**Qué pide la HU-4**
1. Usar el dataset consolidado (CIMA + métricas de la ficha técnica + precios del Nomenclátor).
2. Hacer un EDA con gráficas para entender las distribuciones.
3. Limpieza básica: nulos, tipos de datos y atípicos/anomalías.
4. Reporte final con las conclusiones y las gráficas principales.
5. **Resultado:** un CSV limpio para la HU-5 (K-Means).

**Cómo está organizado**
| Sección | Qué hace | ¿Modifica los datos? |
|---|---|---|
| 1. Carga | Leer el Excel e inspeccionarlo | No |
| 2. Calidad de los datos | Nulos, duplicados, constantes y tipos | No |
| 3. EDA visual | Distribuciones, atípicos, categóricas y correlaciones | No |
| 4. Limpieza | Aplicar todas las decisiones | **Sí** |
| 5. Exportar | Guardar el CSV limpio | — |
| 6. Reporte final | Conclusiones y gráficas clave | — |

Primero se **observa y decide** (secciones 1-3) y después se **limpia todo junto** (sección 4).

## 1. Carga de datos

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df_raw = pd.read_excel("../datos/medicamentos_artritis_reumatoide_con_precios.xlsx")
df = df_raw.copy()          # trabajamos sobre una copia; df_raw queda intacto
print("Filas y columnas:", df.shape)
df.head()

In [ ]:
df.info()

## 2. Calidad de los datos
### 2.1 Resumen por columna

In [ ]:
resumen = pd.DataFrame({
    "tipo": df.dtypes.astype(str),
    "nulos": df.isna().sum(),
    "pct_nulos": (df.isna().mean() * 100).round(1),
    "unicos": df.nunique(),
})
resumen.sort_values("pct_nulos", ascending=False)

### 2.2 Valores nulos

In [ ]:
pct_nulos = resumen.loc[resumen["nulos"] > 0, "pct_nulos"].sort_values()
pct_nulos.plot.barh(figsize=(8, 3), color="indianred", title="% de valores nulos por columna")
plt.xlabel("%")
plt.show()

Las 4 columnas del Nomenclátor tienen el mismo número de nulos. ¿Son **las mismas filas**?
`.all(axis=1)` devuelve True en una fila solo si todas esas columnas son nulas a la vez.

In [ ]:
cols_nomenclator = ["Estado", "Precio de venta al público con IVA",
                    "Tratamiento de larga duración", "Especial control médico"]
sin_nomenclator = df[cols_nomenclator].isna().all(axis=1)
print("Medicamentos sin ningún dato del Nomenclátor:", sin_nomenclator.sum())

pd.crosstab(sin_nomenclator, df["comercializado"], rownames=["sin nomenclátor"])

**Conclusión sobre los nulos**
- **Columnas del Nomenclátor (~48 %):** son las **mismas filas**: medicamentos que no se encontraron al cruzar por `cn` en la HU-3. Una parte no está comercializada (es normal que no figure). La otra parte sí lo está: son sobre todo **biológicos de uso hospitalario** (adalimumab, etanercept…), que no están en el Nomenclátor de farmacia, y, probablemente, casos en los que la HU-1 se quedó con un `cn` de una presentación que no figura. → Crear `en_nomenclator` (0/1).
- **`estado_rev` (~78 %):** el nulo significa **"no aplica"**. El enunciado de la HU-1 lo llama "fecha de revisión", pero en la API de CIMA `rev` es la fecha de **revocación**, y los datos lo confirman (sección 2.4). → Crear `revocado` (0/1).
- **`url_foto_materiales` (~67 %):** no todos los medicamentos tienen foto. No aporta nada al análisis. → Eliminar.
- **`Precio de referencia`:** tiene algunos nulos más que el PVP, y donde existen los dos **valen lo mismo** (sección 3.5). → Nos quedamos solo con el PVP.

### 2.3 Duplicados y columnas constantes

In [ ]:
print("Filas duplicadas:", df.duplicated().sum())
print("cn duplicados:   ", df["cn"].duplicated().sum())

constantes = [col for col in df.columns if df[col].nunique() == 1]
print("Columnas con un único valor:", constantes)

- No hay duplicados: **`cn` identifica cada medicamento**.
- Las columnas constantes no distinguen un medicamento de otro, así que se eliminan:
  - `requiere_receta`: todos requieren receta.
  - `medicamento_huerfano`: ninguno es huérfano.
  - `num_registros_atc`: siempre vale 3.
  - `Especial control médico`: todos los que tienen dato son "NO".
  - `num_excipientes`: siempre 0. Es una **anomalía**: la HU-1 hace `len(med.get("excipientes", []))` y el campo no llega, así que el `[]` por defecto se convierte en 0. Queda como mejora pendiente de la HU-1.

### 2.4 Tipos de datos

In [ ]:
# nregistro: ¿se han perdido ceros a la izquierda? La URL de la ficha técnica conserva el número original
nregistro_url = df["url_html_ficha_tecnica"].str.extract(r"/ft/(\d+)/")[0]   # el número entre /ft/ y /
print("nregistro distinto al de la URL:", (nregistro_url != df["nregistro"].astype(str)).sum())

# Fechas: vienen como milisegundos desde 1970
print(df["estado_aut"].head(3).tolist())

# estado_rev frente a comercializado: ¿revisión o revocación?
pd.crosstab(df["estado_rev"].notna(), df["comercializado"], rownames=["tiene estado_rev"])

| Columna | Problema | Solución (sección 4) |
|---|---|---|
| `cn` | Es un código, no una cantidad; como número pierde los ceros | Texto de 6 dígitos (`zfill(6)`) |
| `nregistro` | Algunos han perdido el 0 inicial | Recuperarlo de la URL |
| `estado_aut` | Fecha en milisegundos | Convertir a fecha → `antiguedad_anios` |
| `estado_rev` | Fecha en ms. Los medicamentos que la tienen están (casi) todos **no comercializados** → es revocación | `revocado` (0/1) |
| `Tratamiento de larga duración` | Texto SI/NO | Binaria 0/1 |

## 3. EDA visual
### 3.1 Variables numéricas

In [ ]:
numericas = ["volumen_informativo_seguridad", "complejidad_desglose_clinico",
             "indicador_riesgo_severo", "Precio de venta al público con IVA"]
df[numericas].describe().round(1)

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 6))
for i, col in enumerate(numericas):
    sns.histplot(df[col], bins=30, ax=axes[0, i])
    sns.boxplot(x=df[col], ax=axes[1, i])
    axes[0, i].set_title(col, fontsize=9)
plt.tight_layout()
plt.show()

print("Asimetría (0 = simétrica, >1 = cola larga a la derecha):")
print(df[numericas].skew().round(2))

- `volumen_informativo_seguridad` es casi simétrica. Está en **miles**, mientras que las demás están en unidades o decenas.
- `complejidad_desglose_clinico`, `indicador_riesgo_severo` y sobre todo el **PVP** tienen una **cola larga a la derecha**: la **media es mayor que la mediana**. La mayoría de los medicamentos son baratos (mediana ≈ 5 €) y unos pocos son caros.
- **Para la HU-5:** como las escalas son muy distintas, hay que **escalar** antes del K-Means; si no, el volumen dominaría las distancias.

### 3.2 Valores atípicos (criterio IQR)
Un valor es atípico si está fuera de **[Q1 − 1,5·IQR, Q3 + 1,5·IQR]**, donde IQR = Q3 − Q1 (lo que dibuja el boxplot).

In [ ]:
for col in numericas:
    q1, q3 = df[col].quantile([0.25, 0.75])
    iqr = q3 - q1
    atipicos = df[(df[col] < q1 - 1.5 * iqr) | (df[col] > q3 + 1.5 * iqr)]
    print(f"{col}: {len(atipicos)} atípicos. Ejemplos: {atipicos['pactivos'].value_counts().head(3).index.tolist()}")

In [ ]:
# El precio más alto: ¿error o real?
df[df["pactivos"] == "LEFLUNOMIDA"][["nombre", "cn", "Precio de venta al público con IVA"]] \
    .sort_values("Precio de venta al público con IVA", ascending=False).head()

- Los atípicos de las métricas de la ficha técnica son **biológicos** (infliximab, rituximab, adalimumab…), con fichas mucho más largas. **Son reales y no se eliminan**: son justo los perfiles que el clustering tiene que separar.
- **Anomalía de precio:** dos leflunomidas de 20 mg cuestan 167,48 €, frente a 53,92 € del resto. Probablemente el `cn` corresponde a un envase más grande. Se conservan y se anotan como pendientes de verificar.

### 3.3 Variables binarias

In [ ]:
binarias = ["comercializado", "generico", "biosimilar", "afecta_conduccion", "triangulo_negro"]
(df[binarias].mean() * 100).sort_values().plot.barh(figsize=(7, 3), title="% de medicamentos con valor 1")
plt.xlabel("%")
plt.show()

- Unos dos tercios están comercializados y más de la mitad son genéricos.
- `biosimilar` (~9 %), `afecta_conduccion` y `triangulo_negro` (~2 %) están **muy desbalanceadas**: son poco frecuentes, pero aportan información clínica, así que se mantienen.

### 3.4 Variables categóricas

In [ ]:
vias = df["vias_administracion"].str.get_dummies(sep=", ")   # una columna 0/1 por vía

fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))
df["forma_farmaceutica_simplificada"].value_counts().sort_values().plot.barh(ax=axes[0], title="Forma farmacéutica")
vias.sum().sort_values().plot.barh(ax=axes[1], title="Vías de administración (separadas)")
df["pactivos"].value_counts().head(10).sort_values().plot.barh(ax=axes[2], title="Top 10 principios activos")
plt.tight_layout()
plt.show()

- **Forma farmacéutica:** dominan los comprimidos y los inyectables. Hay 12 formas, varias con muy pocos casos → se agrupan en 4.
- **Vías:** una celda puede tener **varias vías** ("VÍA INTRAMUSCULAR, VÍA SUBCUTÁNEA"), por eso salen 15 "categorías" que en realidad son combinaciones. Separadas, dominan la oral y la subcutánea → una columna 0/1 por vía principal y `via_otras` para las raras.
- **Principios activos:** metotrexato, ibuprofeno y etoricoxib concentran gran parte del catálogo. Hay sales del mismo fármaco escritas distinto (`METOTREXATO` / `METOTREXATO DISODIO`).

### 3.5 Correlaciones

In [ ]:
cols_corr = numericas + ["Precio de referencia", "num_principios_activos"]
plt.figure(figsize=(8, 6))
sns.heatmap(df[cols_corr].corr(), annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1)
plt.title("Correlación entre variables numéricas")
plt.show()

- **PVP y Precio de referencia tienen correlación 1**: donde existen los dos, son el mismo valor → nos quedamos solo con el **PVP**.
- Las tres métricas de la ficha técnica están correlacionadas positivamente (≈0,6-0,7): una ficha más larga tiene más tablas y más menciones de "grave". No son idénticas, así que se mantienen.

### 3.6 Fecha de autorización

In [ ]:
anio_aut = pd.to_datetime(df["estado_aut"], unit="ms").dt.year
anio_aut.plot.hist(bins=30, figsize=(8, 3), title="Año de autorización")
plt.show()

Hay autorizaciones desde 1958 hasta 2026, y la mitad son posteriores a ~2015 (genéricos y biosimilares). Para el modelo es más útil la **antigüedad en años** que la fecha en sí.

## 4. Limpieza
Aplicamos sobre `df` todas las decisiones de las secciones 2 y 3.

### 4.1 Corregir tipos

In [ ]:
# Identificadores como texto
df["cn"] = df["cn"].astype(str).str.zfill(6)
df["nregistro"] = nregistro_url                       # recupera los ceros perdidos

# Fecha de autorización -> antigüedad en años
# (CIMA guarda la medianoche española en UTC: se convierte a hora de Madrid para no caer en el día anterior)
fecha_aut = pd.to_datetime(df["estado_aut"], unit="ms", utc=True).dt.tz_convert("Europe/Madrid").dt.tz_localize(None)
df["antiguedad_anios"] = ((pd.Timestamp("2026-10-30") - fecha_aut).dt.days / 365.25).round(1)

### 4.2 Tratar los nulos
| Columna | Tratamiento |
|---|---|
| `estado_rev` | → `revocado` (1 si tiene fecha) |
| Columnas del Nomenclátor | → `en_nomenclator` (1 si figura) |
| `Tratamiento de larga duración` | SI/NO → 1/0; si no figura en el Nomenclátor, 0 |
| PVP | Mediana **del mismo principio activo**, y nunca la media global, que inventaría un pico artificial. Se guarda `tiene_pvp` para saber cuáles eran reales |

In [ ]:
df["revocado"] = df["estado_rev"].notna().astype(int)
df["en_nomenclator"] = df["Estado"].notna().astype(int)
df["Estado"] = df["Estado"].fillna("NO FIGURA")

print(df["Tratamiento de larga duración"].value_counts(dropna=False))
df["tratamiento_larga_duracion"] = df["Tratamiento de larga duración"].map({"SI": 1, "NO": 0}).fillna(0).astype(int)

pvp = "Precio de venta al público con IVA"
df["tiene_pvp"] = df[pvp].notna().astype(int)
df["pvp"] = df[pvp].fillna(df.groupby("pactivos")[pvp].transform("median"))
print("PVP nulos antes:", df[pvp].isna().sum(), "| después:", df["pvp"].isna().sum())

Los PVP que siguen nulos son de principios activos **sin ningún precio** (biológicos hospitalarios). No se inventan: se dejan nulos y la HU-5 decide si usa el precio en el K-Means.

### 4.3 Categóricas → numéricas
El K-Means solo trabaja con números, así que cada categoría pasa a ser una columna 0/1.

In [ ]:
# Vías de administración: 4 principales + otras
df["via_oral"] = vias["VÍA ORAL"]
df["via_subcutanea"] = vias["VÍA SUBCUTÁNEA"]
df["via_intravenosa"] = vias["VÍA INTRAVENOSA"]
df["via_intramuscular"] = vias["VÍA INTRAMUSCULAR"]
otras = vias.columns.drop(["VÍA ORAL", "VÍA SUBCUTÁNEA", "VÍA INTRAVENOSA", "VÍA INTRAMUSCULAR"])
df["via_otras"] = vias[otras].max(axis=1)              # 1 si tiene alguna de las vías raras

# Forma farmacéutica: 12 formas -> 4 grupos
forma = df["forma_farmaceutica_simplificada"]
df["forma_oral_solida"] = forma.str.contains("COMPRIMIDO|CAPSULA").astype(int)
df["forma_oral_liquida"] = forma.str.contains("ORAL").astype(int)
df["forma_inyectable"] = forma.str.contains("INYECTABLE").astype(int)
df["forma_rectal"] = forma.str.contains("SUPOSITORIO").astype(int)

grupos = ["forma_oral_solida", "forma_oral_liquida", "forma_inyectable", "forma_rectal"]
print("¿Cada medicamento está en un solo grupo?", (df[grupos].sum(axis=1) == 1).all())

### 4.4 Eliminar columnas sin valor

In [ ]:
a_eliminar = constantes + [
    "estado_aut", "estado_rev",                        # sustituidas por antiguedad_anios y revocado
    "url_html_ficha_tecnica", "url_foto_materiales",   # enlaces, sin valor analítico
    "vias_administracion", "forma_farmaceutica_simplificada",   # ya convertidas a 0/1
    "Tratamiento de larga duración",                   # ya convertida a 0/1
    pvp, "Precio de referencia",                       # sustituidas por pvp (son idénticas)
]
df = df.drop(columns=a_eliminar)
print("Forma final:", df.shape)
df.info()

## 5. Exportar el CSV limpio

In [ ]:
ruta_csv = "../datos/medicamentos_artritis_reumatoide_limpio.csv"
df.to_csv(ruta_csv, index=False, encoding="utf-8-sig")   # utf-8-sig: Excel muestra bien las tildes

# Comprobación: al leerlo, cn y nregistro como texto para no perder los ceros
comprobacion = pd.read_csv(ruta_csv, dtype={"cn": str, "nregistro": str})
print("CSV guardado:", comprobacion.shape)
comprobacion.head()

## 6. Reporte final
### Gráficas principales

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 9))

pct_nulos.plot.barh(ax=axes[0, 0], color="indianred", title="1. % de nulos (datos originales)")
sns.histplot(df["pvp"], bins=30, ax=axes[0, 1]).set_title("2. PVP: cola larga a la derecha")
vias.sum().sort_values().plot.barh(ax=axes[1, 0], title="3. Vías de administración")
sns.heatmap(df_raw[numericas].corr(), annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1,
            ax=axes[1, 1]).set_title("4. Correlación entre métricas")
plt.tight_layout()
plt.show()

### Conclusiones
1. **Dataset:** unos 640 medicamentos y 31 columnas de CIMA, la ficha técnica y el Nomenclátor. No hay duplicados y `cn` es la clave única.
2. **Nulos con significado:** casi la mitad de los medicamentos no figura en el Nomenclátor (no comercializados, biológicos hospitalarios o `cn` de otra presentación). `estado_rev` vacío significa "no revocado". Se han convertido en variables 0/1 en lugar de inventar valores.
3. **Tipos corregidos:** `cn` y `nregistro` como texto (se recuperan los ceros perdidos) y fechas en milisegundos → `antiguedad_anios`.
4. **Columnas eliminadas:** 5 constantes, URLs y columnas redundantes (el Precio de referencia es idéntico al PVP).
5. **Distribuciones:** el precio y las métricas de riesgo tienen cola larga a la derecha. Los atípicos son **biológicos reales** y se conservan.
6. **Anomalías detectadas:** `num_excipientes` siempre 0 (bug de la HU-1), `nregistro` sin ceros iniciales y dos leflunomidas con un precio tres veces mayor.
7. **Resultado:** `medicamentos_artritis_reumatoide_limpio.csv`, con identificadores (texto), variables numéricas y variables 0/1.

### Recomendaciones para la HU-5
- **Escalar** las numéricas (`StandardScaler`) y valorar `log1p` en las de cola larga.
- No usar en el K-Means los identificadores ni el texto (`nregistro`, `cn`, `nombre`, `pactivos`, laboratorios, `dosis`, `Estado`).
- Decidir qué hacer con el PVP de los medicamentos sin precio: excluir la columna o usar solo `tiene_pvp`.